<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Cuadernos/01_Preprocesamiento_Tokenizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Preprocesamiento de texto y tokenización</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Introducción</span>](#introduccion)
- [<span style="color: #005C99;">Limpieza con expresiones regulares</span>](#limpieza-con-expresiones-regulares)
- [<span style="color: #005C99;">Normalización</span>](#normalizacion)
- [<span style="color: #005C99;">Tokenización</span>](#tokenizacion)
- [<span style="color: #005C99;">Stopwords</span>](#stopwords)
- [<span style="color: #005C99;">Stemming</span>](#stemming)
- [<span style="color: #005C99;">Lematización</span>](#lematizacion)
- [<span style="color: #005C99;">Tokenización por subpalabras: BPE</span>](#tokenizacion-por-subpalabras-bpe)
- [<span style="color: #005C99;">Un pipeline completo</span>](#un-pipeline-completo)
- [<span style="color: #005C99;">Referencias</span>](#referencias)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib nltk spacy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Modelo de spaCy para español (pequeño, ~15 MB)
import sys
!{sys.executable} -m spacy download es_core_news_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [3]:
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
import spacy

SEED = 42
np.random.seed(SEED)

for recurso in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(recurso, quiet=True)

nlp = spacy.load("es_core_news_sm")

<a id="introduccion"></a>
<h2 style="color: #007ACC;">Introducción</h2>

Los modelos no trabajan con texto crudo: trabajan con **unidades discretas** (tokens) que luego convertimos en números. El **preprocesamiento** es el conjunto de pasos que transforma el texto original en una secuencia de tokens limpia y consistente.

Un pipeline típico de PLN clásico se ve así:

```text
texto crudo → limpieza → normalización → tokenización → stopwords → stemming / lematización → representación numérica
```

No existe un preprocesamiento "correcto" universal: **cada paso es una decisión de modelado** que depende de la tarea. Por ejemplo, eliminar signos de exclamación puede ser buena idea para clasificar temas de noticias, pero mala idea para detectar sentimiento (*"¡¡Excelente!!"*). Los modelos modernos basados en Transformers, de hecho, hacen muy poco preprocesamiento y delegan el trabajo a un tokenizador por subpalabras.

Trabajaremos con este pequeño corpus de reseñas y publicaciones en español:

In [4]:
corpus = [
    "¡¡Me ENCANTÓ el restaurante!! La comida estaba deliciosa 😍 https://ejemplo.com/resena",
    "@cliente_feliz El servicio fue pésimo, esperamos 45 minutos... NO vuelvo #decepcionado",
    "Los niños jugaban en el parque mientras las niñas corrían por la cancha.",
    "<p>Compré 2 libros por $50.000 y llegaron el 15/03/2024. ¡Muy recomendados!</p>",
    "El Sr. Pérez trabaja en la U. Nacional. Llegó a las 8 a.m. y se fue temprano.",
]
for i, doc in enumerate(corpus):
    print(i, doc)

0 ¡¡Me ENCANTÓ el restaurante!! La comida estaba deliciosa 😍 https://ejemplo.com/resena
1 @cliente_feliz El servicio fue pésimo, esperamos 45 minutos... NO vuelvo #decepcionado
2 Los niños jugaban en el parque mientras las niñas corrían por la cancha.
3 <p>Compré 2 libros por $50.000 y llegaron el 15/03/2024. ¡Muy recomendados!</p>
4 El Sr. Pérez trabaja en la U. Nacional. Llegó a las 8 a.m. y se fue temprano.


<a id="limpieza-con-expresiones-regulares"></a>
<h2 style="color: #007ACC;">Limpieza con expresiones regulares</h2>

Las **expresiones regulares** (módulo `re`) son la herramienta básica para limpiar texto. Los patrones más útiles en PLN son:

| Patrón | Significado | Ejemplo |
|--------|-------------|---------|
| `\d+` | Uno o más dígitos | `2024` |
| `\w+` | Caracteres de palabra (letras, dígitos, `_`) | `cliente_feliz` |
| `\s+` | Espacios en blanco | `"  \n"` |
| `[^...]` | Cualquier carácter que **no** esté en el conjunto | `[^a-z]` |
| `https?://\S+` | URL | `https://ejemplo.com` |
| `@\w+` / `#\w+` | Mención / hashtag | `@cliente_feliz` |
| `<[^>]+>` | Etiqueta HTML | `<p>` |

In [5]:
PATRONES = {
    "url": r"https?://\S+|www\.\S+",
    "html": r"<[^>]+>",
    "mencion": r"@\w+",
    "hashtag": r"#(\w+)",       # conservamos la palabra, quitamos el símbolo
    "numero": r"\d+(?:[.,/]\d+)*",
}


def limpiar(texto):
    texto = re.sub(PATRONES["url"], " ", texto)
    texto = re.sub(PATRONES["html"], " ", texto)
    texto = re.sub(PATRONES["mencion"], " ", texto)
    texto = re.sub(PATRONES["hashtag"], r"\1", texto)
    texto = re.sub(PATRONES["numero"], " <NUM> ", texto)
    return re.sub(r"\s+", " ", texto).strip()


for doc in corpus:
    print(limpiar(doc))

¡¡Me ENCANTÓ el restaurante!! La comida estaba deliciosa 😍
El servicio fue pésimo, esperamos <NUM> minutos... NO vuelvo decepcionado
Los niños jugaban en el parque mientras las niñas corrían por la cancha.
Compré <NUM> libros por $ <NUM> y llegaron el <NUM> . ¡Muy recomendados!
El Sr. Pérez trabaja en la U. Nacional. Llegó a las <NUM> a.m. y se fue temprano.


Fíjate en dos decisiones:

- Los números se reemplazaron por un **token especial** `<NUM>` en lugar de eliminarlos. Así el modelo sabe que *había* un número sin tener que aprender cada valor por separado.
- Los hashtags conservan la palabra (`decepcionado`), porque suele llevar información útil.

Los emojis también son informativos para sentimiento (😍). Podemos detectarlos por su rango Unicode:

In [6]:
patron_emoji = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF]")
print(patron_emoji.findall(corpus[0]))

['😍']


<a id="normalizacion"></a>
<h2 style="color: #007ACC;">Normalización</h2>

**Normalizar** es llevar variantes de una misma palabra a una forma canónica, para que el modelo no las trate como palabras distintas:

- **Minúsculas:** *ENCANTÓ*, *Encantó* y *encantó* → *encantó*. Se pierde la distinción de nombres propios (*Pérez*, *Nacional*), que puede ser importante para NER.
- **Acentos y diacríticos:** usuarios que escriben *pesimo* en vez de *pésimo*. Quitar tildes unifica ambas formas, pero en español puede **cambiar el significado**: *año* / *ano*, *papá* / *papa*, *está* / *esta*.
- **Caracteres repetidos:** *¡¡¡buenísimooo!!!* → *¡buenísimo!*.
- **Unicode:** el mismo carácter visual puede tener representaciones distintas (`é` como un solo código o como `e` + tilde combinada).

In [7]:
def quitar_tildes(texto):
    """Elimina diacríticos (á -> a, ü -> u) pero conserva la ñ."""
    descompuesto = unicodedata.normalize("NFD", texto)           # 'á' -> 'a' + tilde combinada
    sin_marcas = re.sub(r"(?<![nN])[\u0300-\u036f]", "", descompuesto)
    return unicodedata.normalize("NFC", sin_marcas)


def reducir_repeticiones(texto, max_rep=2):
    """buenísimooooo -> buenísimoo ; !!!! -> !!"""
    return re.sub(r"(.)\1{%d,}" % max_rep, r"\1" * max_rep, texto)


ejemplos = ["Pésimo SERVICIO", "El año pasado", "Buenísimooooo!!!!", "Canción del niño"]
pd.DataFrame({
    "original": ejemplos,
    "minúsculas": [e.lower() for e in ejemplos],
    "sin tildes": [quitar_tildes(e.lower()) for e in ejemplos],
    "sin repeticiones": [reducir_repeticiones(e.lower()) for e in ejemplos],
})

,original,minúsculas,sin tildes,sin repeticiones
0,Pésimo SERVICIO,pésimo servicio,pesimo servicio,pésimo servicio
1,El año pasado,el año pasado,el año pasado,el año pasado
2,Buenísimooooo!!!!,buenísimooooo!!!!,buenisimooooo!!!!,buenísimoo!!
3,Canción del niño,canción del niño,cancion del niño,canción del niño


In [8]:
# Dos cadenas que se ven iguales pero no lo son
a = "canción"                       # 'ó' precompuesta (1 carácter)
b = "cancio\u0301n"                 # 'o' + tilde combinada (2 caracteres)
print(a, b, a == b, len(a), len(b))
print("Tras normalizar NFC:", unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC", b))

canción canción False 7 8
Tras normalizar NFC: True


<a id="tokenizacion"></a>
<h2 style="color: #007ACC;">Tokenización</h2>

**Tokenizar** es dividir el texto en unidades mínimas (*tokens*). Parece trivial, pero no lo es:

- ¿*U. Nacional* es una o dos oraciones? ¿*Sr.* termina la oración?
- ¿*$50.000* es un token, dos o tres?
- ¿*¡Muy* debe separarse en `¡` y `Muy`?
- En idiomas como el chino no hay espacios entre palabras.

Comparemos varias estrategias.

In [9]:
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer

texto = corpus[3] + " " + corpus[4]
texto = re.sub(PATRONES["html"], " ", texto).strip()

tokenizadores = {
    "split()": texto.split(),
    "regex \\w+": re.findall(r"\w+", texto),
    "NLTK word_tokenize": word_tokenize(texto, language="spanish"),
    "spaCy": [t.text for t in nlp(texto)],
}
for nombre, toks in tokenizadores.items():
    print(f"{nombre:<20} ({len(toks):>2} tokens): {toks}\n")

split()              (28 tokens): ['Compré', '2', 'libros', 'por', '$50.000', 'y', 'llegaron', 'el', '15/03/2024.', '¡Muy', 'recomendados!', 'El', 'Sr.', 'Pérez', 'trabaja', 'en', 'la', 'U.', 'Nacional.', 'Llegó', 'a', 'las', '8', 'a.m.', 'y', 'se', 'fue', 'temprano.']

regex \w+            (32 tokens): ['Compré', '2', 'libros', 'por', '50', '000', 'y', 'llegaron', 'el', '15', '03', '2024', 'Muy', 'recomendados', 'El', 'Sr', 'Pérez', 'trabaja', 'en', 'la', 'U', 'Nacional', 'Llegó', 'a', 'las', '8', 'a', 'm', 'y', 'se', 'fue', 'temprano']

NLTK word_tokenize   (35 tokens): ['Compré', '2', 'libros', 'por', '$', '50.000', 'y', 'llegaron', 'el', '15/03/2024', '.', '¡Muy', 'recomendados', '!', 'El', 'Sr.', 'Pérez', 'trabaja', 'en', 'la', 'U', '.', 'Nacional', '.', 'Llegó', 'a', 'las', '8', 'a.m', '.', 'y', 'se', 'fue', 'temprano', '.']

spaCy                (36 tokens): ['Compré', '2', 'libros', 'por', '$', '50.000', 'y', 'llegaron', 'el', '15/03/2024', '.', '¡', 'Muy', 'recomendados', '!',

- `split()` deja la puntuación pegada a las palabras (`recomendados!</p>`).
- `\w+` elimina toda la puntuación, incluidos los signos que pueden importar.
- NLTK y spaCy usan reglas específicas del idioma: separan `¡`, mantienen `$50.000` y reconocen abreviaturas.

Para texto de redes sociales conviene un tokenizador especializado que respete menciones, hashtags y emojis:

In [10]:
tweet = corpus[1] + " 😡😡 :("
print("word_tokenize: ", word_tokenize(tweet, language="spanish"))
print("TweetTokenizer:", TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True).tokenize(tweet))

word_tokenize:  ['@', 'cliente_feliz', 'El', 'servicio', 'fue', 'pésimo', ',', 'esperamos', '45', 'minutos', '...', 'NO', 'vuelvo', '#', 'decepcionado', '😡😡', ':', '(']
TweetTokenizer: ['el', 'servicio', 'fue', 'pésimo', ',', 'esperamos', '45', 'minutos', '...', 'no', 'vuelvo', '#decepcionado', '😡', '😡', ':(']


<h3 style="color: #007ACC;">Segmentación en oraciones</h3>

In [11]:
texto = corpus[4]
print("Partir en '.':  ", [s.strip() for s in texto.split(".") if s.strip()])
print("NLTK (Punkt):   ", sent_tokenize(texto, language="spanish"))
print("spaCy:          ", [s.text for s in nlp(texto).sents])

Partir en '.':   ['El Sr', 'Pérez trabaja en la U', 'Nacional', 'Llegó a las 8 a', 'm', 'y se fue temprano']
NLTK (Punkt):    ['El Sr. Pérez trabaja en la U.', 'Nacional.', 'Llegó a las 8 a.m.', 'y se fue temprano.']
spaCy:           ['El Sr. Pérez trabaja en la U. Nacional.', 'Llegó a las 8 a.m.', 'y se fue temprano.']


Partir ingenuamente por puntos rompe las abreviaturas (*Sr.*, *U.*, *a.m.*). Los segmentadores entrenados lo hacen mejor, aunque ninguno es perfecto: siempre conviene **inspeccionar los resultados** sobre los datos reales.

<a id="stopwords"></a>
<h2 style="color: #007ACC;">Stopwords</h2>

Las **stopwords** (palabras vacías) son palabras muy frecuentes y con poco contenido temático: artículos, preposiciones, conjunciones (*el, la, de, que, y, en*...). Por la ley de Zipf ocupan buena parte del texto, así que eliminarlas reduce el tamaño de las representaciones.

⚠️ **Cuidado:** las listas estándar incluyen palabras que pueden ser cruciales según la tarea. La lista de NLTK para español incluye *no*, *ni*, *muy*, *nada*, *sin*: eliminarlas convertiría *"no me gustó"* en *"gustó"*, invirtiendo el sentimiento.

In [12]:
from nltk.corpus import stopwords

stop_es = set(stopwords.words("spanish"))
print(f"{len(stop_es)} stopwords en español. Algunas: {sorted(stop_es)[:15]}")
print("Palabras de negación/intensidad en la lista:", sorted(stop_es & {"no", "ni", "nada", "nunca", "muy", "sin", "poco"}))

313 stopwords en español. Algunas: ['a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde']
Palabras de negación/intensidad en la lista: ['muy', 'nada', 'ni', 'no', 'poco', 'sin']


In [13]:
NEGACIONES = {"no", "ni", "nada", "nunca", "sin", "muy", "poco"}
stop_sentimiento = stop_es - NEGACIONES

frase = "no me gustó nada la comida, estaba muy fría"
tokens = [t for t in word_tokenize(frase, language="spanish") if t.isalpha()]
print("Lista completa:     ", [t for t in tokens if t not in stop_es])
print("Sin quitar negación:", [t for t in tokens if t not in stop_sentimiento])

Lista completa:      ['gustó', 'comida', 'fría']
Sin quitar negación: ['no', 'gustó', 'nada', 'comida', 'muy', 'fría']


<a id="stemming"></a>
<h2 style="color: #007ACC;">Stemming</h2>

El **stemming** recorta sufijos con reglas heurísticas para obtener una raíz (*stem*). Es rápido y no requiere diccionario, pero la raíz no siempre es una palabra real.

- **Porter** (1980): el algoritmo clásico, solo para inglés.
- **Snowball** (Porter 2): familia de stemmers para muchos idiomas, incluido el español.

In [14]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
print("Porter (inglés):", {w: porter.stem(w) for w in ["running", "runs", "ran", "easily", "studies", "university"]})

snowball = SnowballStemmer("spanish")
palabras = ["corriendo", "corrían", "corrió", "niños", "niñas", "comunicación", "comunicar", "comunidad", "estaba", "fue", "es"]
pd.DataFrame({"palabra": palabras, "stem": [snowball.stem(p) for p in palabras]}).T

Porter (inglés): {'running': 'run', 'runs': 'run', 'ran': 'ran', 'easily': 'easili', 'studies': 'studi', 'university': 'univers'}


,0,1,2,3,4,5,6,7,8,9,10
palabra,corriendo,corrían,corrió,niños,niñas,comunicación,comunicar,comunidad,estaba,fue,es
stem,corr,corr,corr,niñ,niñ,comun,comunic,comun,estab,fue,es


Observa dos tipos de error:

- **Sobre-stemming:** palabras con significados distintos que terminan en la misma raíz (*comunicación* y *comunidad* → `comun`).
- **Sub-stemming:** formas del mismo verbo que quedan separadas (*es*, *fue*, *estaba* no se unifican porque son irregulares).

<a id="lematizacion"></a>
<h2 style="color: #007ACC;">Lematización</h2>

La **lematización** reduce cada palabra a su **lema**, la forma que aparece en el diccionario (infinitivo para verbos, masculino singular para adjetivos y sustantivos). A diferencia del stemming, usa el análisis morfológico y el contexto gramatical, así que maneja formas irregulares.

spaCy lematiza en español junto con el etiquetado gramatical:

In [15]:
doc = nlp("Los niños jugaban en el parque mientras las niñas corrían. Fue un día que estuvimos felices.")

pd.DataFrame(
    [(t.text, t.lemma_, snowball.stem(t.text.lower()), t.pos_) for t in doc if not t.is_punct],
    columns=["token", "lema (spaCy)", "stem (Snowball)", "categoría"],
)

,token,lema (spaCy),stem (Snowball),categoría
0,Los,el,los,DET
1,niños,niño,niñ,NOUN
2,jugaban,jugar,jug,VERB
3,en,en,en,ADP
4,el,el,el,DET
5,parque,parque,parqu,NOUN
6,mientras,mientras,mientr,SCONJ
7,las,el,las,DET
8,niñas,niña,niñ,NOUN
9,corrían,correr,corr,VERB


| | Stemming | Lematización |
|---|----------|--------------|
| Resultado | Raíz recortada (`jug`, `comun`) | Palabra del diccionario (`jugar`) |
| Velocidad | Muy rápido | Más lento (requiere modelo) |
| Formas irregulares | Falla (*fue* ≠ *ser*) | Las maneja (*fue* → *ser* o *ir*, según contexto) |
| Recursos | Solo reglas | Diccionario / modelo del idioma |
| Uso típico | Búsqueda, clasificación con mucho texto | Cuando la interpretabilidad importa o el idioma es muy flexivo (como el español) |

Los lematizadores tampoco son perfectos: el modelo pequeño de spaCy puede fallar con formas poco frecuentes (revisa la tabla anterior). Los modelos más grandes (`es_core_news_md`, `es_core_news_lg`) son más precisos a cambio de más memoria y tiempo.

<a id="tokenizacion-por-subpalabras-bpe"></a>
<h2 style="color: #007ACC;">Tokenización por subpalabras: BPE</h2>

Los enfoques anteriores trabajan con **palabras completas**, lo que deja dos problemas sin resolver:

1. **Palabras fuera del vocabulario (OOV):** una palabra nueva (*"desparchado"*, *"reutilizabilidad"*) no tiene representación.
2. **Vocabularios enormes:** por la ley de Zipf, cubrir el idioma requiere cientos de miles de palabras.

La **tokenización por subpalabras** resuelve ambos: las palabras frecuentes se mantienen enteras y las raras se descomponen en fragmentos frecuentes (*des + parch + ado*). Es la base de todos los modelos modernos (BERT usa WordPiece; GPT usa BPE a nivel de bytes).

**Byte Pair Encoding (BPE)** (Sennrich et al., 2016):

1. Empieza con un vocabulario de caracteres individuales (cada palabra termina con `</w>`).
2. Cuenta todos los pares de símbolos adyacentes en el corpus.
3. Fusiona el par más frecuente en un nuevo símbolo.
4. Repite los pasos 2–3 hasta alcanzar el número de fusiones deseado.

Implementémoslo desde cero:

In [16]:
def contar_pares(vocab):
    pares = Counter()
    for palabra, freq in vocab.items():
        simbolos = palabra.split()
        for a, b in zip(simbolos, simbolos[1:]):
            pares[(a, b)] += freq
    return pares


def fusionar(par, vocab):
    # (?<!\S) y (?!\S) evitan fusionar dentro de otros símbolos
    patron = re.compile(r"(?<!\S)" + re.escape(" ".join(par)) + r"(?!\S)")
    return {patron.sub("".join(par), palabra): freq for palabra, freq in vocab.items()}


def entrenar_bpe(textos, n_fusiones):
    frec_palabras = Counter(w for t in textos for w in t.lower().split())
    vocab = {" ".join(w) + " </w>": f for w, f in frec_palabras.items()}
    fusiones = []
    for _ in range(n_fusiones):
        pares = contar_pares(vocab)
        if not pares:
            break
        mejor = pares.most_common(1)[0][0]
        vocab = fusionar(mejor, vocab)
        fusiones.append(mejor)
    return fusiones


textos_bpe = [
    "jugar jugando jugamos jugaron jugador jugadores",
    "cantar cantando cantamos cantaron cantante cantantes",
    "bailar bailando bailamos bailaron bailarín",
    "caminar caminando caminamos caminaron",
] * 3

fusiones = entrenar_bpe(textos_bpe, n_fusiones=25)
print("Primeras fusiones aprendidas:")
for i, f in enumerate(fusiones[:12], 1):
    print(f"{i:>2}. {f[0]} + {f[1]} -> {''.join(f)}")

Primeras fusiones aprendidas:
 1. a + n -> an
 2. a + r -> ar
 3. a + m -> am
 4. an + t -> ant
 5. j + u -> ju
 6. ju + g -> jug
 7. d + o -> do
 8. s + </w> -> s</w>
 9. c + ant -> cant
10. n + </w> -> n</w>
11. b + a -> ba
12. ba + i -> bai


In [17]:
def tokenizar_bpe(palabra, fusiones):
    simbolos = list(palabra.lower()) + ["</w>"]
    for a, b in fusiones:  # se aplican en el mismo orden en que se aprendieron
        i = 0
        while i < len(simbolos) - 1:
            if simbolos[i] == a and simbolos[i + 1] == b:
                simbolos[i:i + 2] = [a + b]
            else:
                i += 1
    return simbolos


# Palabras que NO estaban en el corpus de entrenamiento
for palabra in ["jugando", "nadando", "pintaron", "jugadora"]:
    print(f"{palabra:<10} -> {tokenizar_bpe(palabra, fusiones)}")

jugando    -> ['jug', 'ando</w>']
nadando    -> ['n', 'a', 'd', 'ando</w>']
pintaron   -> ['p', 'i', 'n', 't', 'aron</w>']
jugadora   -> ['jugado', 'r', 'a', '</w>']


BPE descubrió por sí solo los sufijos verbales del español (`ando`, `amos`, `aron`) y puede tokenizar palabras que nunca vio (*nadando*, *pintaron*) combinando fragmentos conocidos. En el módulo 4 usaremos los tokenizadores BPE y WordPiece ya entrenados de Hugging Face.

<a id="un-pipeline-completo"></a>
<h2 style="color: #007ACC;">Un pipeline completo</h2>

Juntemos todo en una función configurable. Hacerla **parametrizable** es una buena práctica: permite comparar variantes de preprocesamiento como si fueran hiperparámetros del modelo.

In [18]:
# Le indicamos a spaCy que <NUM> es un único token
nlp.tokenizer.add_special_case("<NUM>", [{"ORTH": "<NUM>"}])


def preprocesar(texto, minusculas=True, tildes=False, stopwords_=None, metodo=None):
    """Pipeline de preprocesamiento.

    metodo: None, "stem" o "lema".
    stopwords_: conjunto de stopwords a eliminar (None para no eliminar).
    """
    if minusculas:
        texto = texto.lower()  # antes de spaCy: "ENCANTÓ" en mayúsculas confunde al lematizador
    texto = reducir_repeticiones(limpiar(texto))
    tokens = []
    # spaCy analiza el texto con tildes: le ayudan a lematizar. Las quitamos al final.
    for t in nlp(texto):
        if t.is_punct or t.is_space:
            continue
        if t.text == "<NUM>":
            tokens.append(t.text)
            continue
        if stopwords_ and t.text.lower() in stopwords_:
            continue
        palabra = t.lemma_ if metodo == "lema" else t.text
        if metodo == "stem":
            palabra = snowball.stem(palabra)
        if not tildes:
            palabra = quitar_tildes(palabra)
        tokens.append(palabra)
    return tokens


for doc in corpus[:3]:
    print(preprocesar(doc, stopwords_=stop_sentimiento, metodo="lema"))

['encantar', 'restaurante', 'comida', 'delicios', '😍']
['servicio', 'pesimo', 'esperar', '<NUM>', 'minuto', 'no', 'volver', 'decepcionado']
['niño', 'jugar', 'parque', 'mientras', 'niña', 'correr', 'cancha']


¿Cuánto reduce cada paso el tamaño del vocabulario? Comparemos las variantes sobre nuestro corpus de ejemplo; con un corpus real la reducción es mucho mayor.

In [19]:
variantes = {
    "solo tokenizar": dict(minusculas=False, tildes=True),
    "+ minúsculas": dict(tildes=True),
    "+ sin tildes": dict(),
    "+ stopwords": dict(stopwords_=stop_sentimiento),
    "+ stemming": dict(stopwords_=stop_sentimiento, metodo="stem"),
    "+ lematización": dict(stopwords_=stop_sentimiento, metodo="lema"),
}

tamanos = {nombre: len({tok for doc in corpus for tok in preprocesar(doc, **kw)}) for nombre, kw in variantes.items()}
pd.Series(tamanos, name="tamaño del vocabulario").to_frame()

,tamaño del vocabulario
solo tokenizar,48
+ minúsculas,46
+ sin tildes,46
+ stopwords,34
+ stemming,32
+ lematización,33


**Resumen de buenas prácticas:**

1. Inspecciona tus datos antes de decidir qué limpiar.
2. Trata cada paso de preprocesamiento como un hiperparámetro y evalúalo con validación.
3. Ajusta las listas de stopwords a la tarea (¡las negaciones importan en sentimiento!).
4. Para español, la lematización suele funcionar mejor que el stemming porque el idioma es muy flexivo.
5. Aplica **exactamente el mismo** preprocesamiento en entrenamiento y en producción.

En el siguiente cuaderno daremos un paso más allá de los tokens: asignaremos a cada palabra su categoría gramatical y resolveremos ambigüedades de significado.

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Bird, S., Klein, E. & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. [nltk.org/book](https://www.nltk.org/book/)
- Documentación de spaCy. [spacy.io/usage](https://spacy.io/usage)
- Sennrich, R., Haddow, B. & Birch, A. (2016). Neural Machine Translation of Rare Words with Subword Units. *ACL*.
- Porter, M. F. (1980). An algorithm for suffix stripping. *Program*, 14(3), 130–137.